# 1.embedding的调用和使用

## 1.1 embedding的获取

In [2]:
from langchain_huggingface import HuggingFaceEmbeddings

emb = HuggingFaceEmbeddings(
    model_name = "sentence-transformers/all-MiniLM-L6-v2",                 
    model_kwargs={"device": "cuda"},
    encode_kwargs={"normalize_embeddings": True}
    # 对生成的向量做 L2 归一化，这样 余弦相似度 = 向量点积，便于检索。
)

e:\miniconda\envs\langchain2\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6387.04it/s]


## 1.2 embedding的使用

In [88]:
from langchain_core.documents import Document

# emb.embed_query("什么是RAG") 

docs = ["什么是RAG", "RAG 是一种结合了检索和生成的自然语言处理技术。"]
docs2 = [Document(page_content="什么是RAG", metadata={"source": "example.txt"}),
        Document(page_content="RAG 是一种结合了检索和生成的自然语言处理技术。", metadata={"source": "example.txt"})]
"""
这里可以区分一下Document和document的区别
前者是langchain的Document类，可以携带metadata等信息 需要import
数据结构为Document(page_content='xxx', metadata={'source': 'example.txt'})
后者是普通字符串列表。

embed_document方法既支持字符串列表，也支持组合为list的Document对象列表，后者可以携带更多的上下文信息，便于后续检索和使用。
"""
emb.embed_documents(docs)

[[-0.038526467978954315,
  0.05861826613545418,
  0.07145114243030548,
  -0.0102588701993227,
  -0.021027501672506332,
  0.026349792256951332,
  0.0993429571390152,
  0.011616144329309464,
  0.024152683094143867,
  -0.038080114871263504,
  0.1135854423046112,
  -0.12874917685985565,
  0.09167085587978363,
  -0.043642446398735046,
  -0.03067154809832573,
  0.057399604469537735,
  0.03972840681672096,
  0.12299804389476776,
  -0.01743190735578537,
  0.011055969633162022,
  -0.058708932250738144,
  0.03324883058667183,
  0.013433156535029411,
  0.02759382128715515,
  0.005614979658275843,
  -0.02044895850121975,
  -0.013871285133063793,
  0.018947090953588486,
  0.042012669146060944,
  -0.0060303653590381145,
  -0.026172902435064316,
  0.036803118884563446,
  -0.05874105915427208,
  -0.04365772381424904,
  0.03098628856241703,
  0.02430526353418827,
  -0.029921822249889374,
  0.011227755807340145,
  0.03685460612177849,
  0.03629891201853752,
  -0.02743804268538952,
  -0.02043356560170650

# 2.向量数据库

In [ ]:
## 2.1向量数据库的创建

In [ ]:
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
emb = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
# 连接或创建collection，不立刻导入数据
db = Chroma(
    persist_directory="./chroma_db",
    collection_name="my_collection",
    embedding_function=emb
)
"""
创建或者加载，里面的chroma.sqlite3是SQLite数据库文件，保存向量collection的结构等配置信息
注意！这里的db是对应 collection的对象，不是整个数据库的对象，Chroma类是collection的封装。
一个db对象只操作当前指定的collection
"""
# 创建 collection 并立即写入纯文本
texts = ["文本1", "文本2"]
db = Chroma.from_texts(
    texts,
    embedding=emb,
    persist_directory="./chroma_db_from_texts",
    collection_name="my_collection"
)
docs = [
    Document(page_content="文本1", metadata={"source": "file1.txt"}),
    Document(page_content="文本2", metadata={"source": "file2.txt"})
]

# 创建 collection 并立即写入 Document
db = Chroma.from_documents(
    docs,
    embedding=emb,
    persist_directory="./chroma_db_from_documents",
    collection_name="my_collection"
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9525.78it/s]


In [ ]:
print(db._collection_name)
print(db._collection.count()) # 返回当前collection的记录数 0表示空的collection 
"""
可以通过db._collection访问内部的collection
创建或者加载，里面的chroma.sqlite3是SQLite数据库文件，保存向量collection的结构等配置信息

"""

my_collection
0


In [40]:
"""
当前langchain没有封装查看collection的
只能用原生Chroma client来实现
"""
import chromadb

client = chromadb.PersistentClient(path="./chroma_db")
collections = client.list_collections()
print([c.name for c in collections])

['my_collection']


In [ ]:
""" 
清空整个collection
"""
client.delete_collection("my_collection")


## 2.2 向量数据库的真实使用场景

In [ ]:
docs = [Document(page_content="什么是RAG", metadata={"source": "example.txt"}),
        Document(page_content="RAG 是一种结合了检索和生成的自然语言处理技术。", metadata={"source": "example.txt"})]
ids = ["doc1", "doc2"]
ids = [f"doc{i}" for i in range(len(docs))]
documents = [doc.page_content for doc in docs]
metadatas = [doc.metadata for doc in docs]

# 向已有 collection 增加文档，推荐主流程用这个
db = Chroma(
    persist_directory="./chroma_db",
    collection_name="my_collection",
    embedding_function=emb
)

db.add_documents(docs)
"""
db.add_documents(docs, ids=ids) 也可以组织id
"""

db._collection.add(
    ids=ids,
    documents=documents,
    metadatas=metadatas
)
"""
Chroma.add_documents是langchain封装好的方法，自动生成ids并调用collection.add方法将文档添加到数据库中。
collection.add是底层方法，需要手动生成ids并传入文档和元数据。两者的区别在于前者更方便快捷，后者提供了更多的控制权。
"""

'\nChroma.add_documents是langchain封装好的方法，自动生成ids并调用collection.add方法将文档添加到数据库中。\ncollection.add是底层方法，需要手动生成ids并传入文档和元数据。两者的区别在于前者更方便快捷，后者提供了更多的控制权。\n'

### TODO

稳定 ID 和去重 去重？
ids = [f"example_{i}" for i in range(len(docs))]
db.add_documents(docs, ids=ids)

清空/重建collection

查看collection的数据规模和抽样数据 
什么是分页？

In [ ]:
new_doc = Document(
    page_content="RAG 是检索增强生成。",
    metadata={"source": "example.txt"}
)

db.add_documents([new_doc], ids=["example_0"])

updated_doc = Document(
    page_content="RAG 是 Retrieval-Augmented Generation，用检索结果增强大模型回答。",
    metadata={"source": "example.txt"}
)

db.update_documents(ids=["example_0"], documents=[updated_doc])
db.delete(ids=["example_0"])



In [16]:
print(db._collection_name)
print(db._collection.count()) # 返回当前collection的记录数 0表示空的collection 
"""
可以通过db._collection访问内部的collection
创建或者加载，里面的chroma.sqlite3是SQLite数据库文件，保存向量collection的结构等配置信息

"""

my_collection
6


'\n可以通过db._collection访问内部的collection\n创建或者加载，里面的chroma.sqlite3是SQLite数据库文件，保存向量collection的结构等配置信息\n\n'

In [35]:
all_data = db._collection.get(
    include=["documents", "metadatas", "embeddings"]
)
for doc_id, doc_text, meta , embedding in zip(all_data["ids"], all_data["documents"], all_data["metadatas"], all_data["embeddings"]):
    print(f"{doc_id}: {doc_text[:30]}..., metadata={meta}, embedding={embedding[:5]}...")

c8d801d0-9f50-412f-ac32-c6c6e4f7ff6a: 什么是RAG..., metadata={'source': 'example.txt'}, embedding=[-0.03852647  0.05861827  0.07145114 -0.01025887 -0.0210275 ]...
0e80caec-0699-4de3-98e2-a118e1179347: RAG 是一种结合了检索和生成的自然语言处理技术。..., metadata={'source': 'example.txt'}, embedding=[ 0.00577979  0.06571506  0.02652241 -0.02234667  0.0312372 ]...
bddf5df9-0839-4e98-96b0-7d497790496d: 什么是RAG..., metadata={'source': 'example.txt'}, embedding=[-0.03852647  0.05861827  0.07145114 -0.01025887 -0.0210275 ]...
8d4e9ee5-caae-4645-bd1c-96e43c0fdcfd: RAG 是一种结合了检索和生成的自然语言处理技术。..., metadata={'source': 'example.txt'}, embedding=[ 0.00577979  0.06571506  0.02652241 -0.02234667  0.0312372 ]...
doc0: 什么是RAG..., metadata={'source': 'example.txt'}, embedding=[-0.03852651  0.05861822  0.07145113 -0.01025887 -0.02102752]...
doc1: RAG 是一种结合了检索和生成的自然语言处理技术。..., metadata={'source': 'example.txt'}, embedding=[ 0.00577986  0.06571502  0.0265224  -0.02234666  0.03123729]...


In [26]:
result = db._collection.get(ids=["c8d801d0-9f50-412f-ac32-c6c6e4f7ff6a"])
print(result)

{'ids': ['c8d801d0-9f50-412f-ac32-c6c6e4f7ff6a'], 'embeddings': None, 'documents': ['什么是RAG'], 'uris': None, 'included': ['metadatas', 'documents'], 'data': None, 'metadatas': [{'source': 'example.txt'}]}


In [ ]:
# 3. 基础向量检索

In [ ]:
# 3.1 基础检索

In [ ]:
query_text = "RAG是什么"

results = db._collection.query(
    query_texts=[query_text], # 可以批量
    n_results=3,
    include=["documents","metadatas","distances"]
)

# 查看结果
for doc, meta, dist , ids in zip(results["documents"][0], results["metadatas"][0], results["distances"][0], results["ids"][0]):
    print(f"{doc[:50]}..., metadata={meta}, similarity={dist}, id={ids}")

什么是RAG..., metadata={'source': 'example.txt'}, similarity=0.041219212114810944, id=c8d801d0-9f50-412f-ac32-c6c6e4f7ff6a
什么是RAG..., metadata={'source': 'example.txt'}, similarity=0.041219212114810944, id=bddf5df9-0839-4e98-96b0-7d497790496d
什么是RAG..., metadata={'source': 'example.txt'}, similarity=0.04121924936771393, id=doc0


In [85]:
"""
db.similaruty_seach                 是简单的top-k
db,similarity_search_with_score     是top-k + 分数
db.max_marginal_relevance_search    是top-k + 分数 + 多样性 mmr

filter参数就是按照metedata做过滤
filter = {
    "source": "example.txt",  #  metedata的类别 : metedata的数值
    "" : ""          
}
"""

docs = db.similarity_search(
    query="RAG是什么",
    k=3,
    # include=["documents", "metadatas", "distances"],
    filter={"source": "example.txt"}  # 只搜索 metadata 中 source=example.txt 的文档
)
print(docs)
print(docs[0].page_content) # 说明了docs是一个list列表，list中存储的是Document对象
"""
for doc, meta in zip(docs.page_content, docs.metadata):
    print(f"{doc[:50]}..., metadata={meta}")
报错是因为Chroma.similarity_search方法默认返回Document对象列表，也就是一个List[Document]，
在Documet层面 才能通过doc.page_content和doc.metadata来访问内容和元数据
所以正确做法是先迭代列表里的每个Doeument对象，访问每个对象的属性
"""
# for doc, meta in zip(docs.page_content, docs.metadata):
#     print(f"{doc[:50]}..., metadata={meta}")

for doc in docs:
    print(f"{doc.page_content[:50]}..., metadata={doc.metadata}, id={doc.id}")

[Document(id='c8d801d0-9f50-412f-ac32-c6c6e4f7ff6a', metadata={'source': 'example.txt'}, page_content='什么是RAG'), Document(id='bddf5df9-0839-4e98-96b0-7d497790496d', metadata={'source': 'example.txt'}, page_content='什么是RAG'), Document(id='doc0', metadata={'source': 'example.txt'}, page_content='什么是RAG')]
什么是RAG
什么是RAG..., metadata={'source': 'example.txt'}, id=c8d801d0-9f50-412f-ac32-c6c6e4f7ff6a
什么是RAG..., metadata={'source': 'example.txt'}, id=bddf5df9-0839-4e98-96b0-7d497790496d
什么是RAG..., metadata={'source': 'example.txt'}, id=doc0


In [ ]:
query_text = "RAG是什么"
docs = db.similarity_search_with_score(
    query=query_text,
    k=3,
    include=["documents", "metadatas", "distances"]
)
print(docs)
# for doc in docs:
#     for Document, score in doc:
#     print(f"{Document.page_content[:50]}..., metadata={Document.metadata}, similarity={score}, id={Document.id}")

tuple = docs[0]  
print(tuple)

"""
docs是一个List，每个元素是一个tuple，tuple的第一个元素是Document对象，第二个元素是相似度分数。
所以方法一：for doc in docs，这里的doc是一个tuple，可以通过doc[0]访问Document对象，doc[1]访问相似度分数。
方法二：for doc_obj, score in docs，把doc_obj, score 看成doc的组成部分
"""
for doc_obj, score in docs:   # 这里直接把元组解包成 doc_obj 和 score
    print(f"{doc_obj.page_content[:50]}..., metadata={doc_obj.metadata}, similarity={score}, id={doc_obj.id}")

for doc in docs:
    print(doc[0].page_content[:50], doc[0].metadata, doc[1])

[(Document(id='c8d801d0-9f50-412f-ac32-c6c6e4f7ff6a', metadata={'source': 'example.txt'}, page_content='什么是RAG'), 0.041219234466552734), (Document(id='bddf5df9-0839-4e98-96b0-7d497790496d', metadata={'source': 'example.txt'}, page_content='什么是RAG'), 0.041219234466552734), (Document(id='doc0', metadata={'source': 'example.txt'}, page_content='什么是RAG'), 0.04121926799416542)]
(Document(id='c8d801d0-9f50-412f-ac32-c6c6e4f7ff6a', metadata={'source': 'example.txt'}, page_content='什么是RAG'), 0.041219234466552734)
什么是RAG..., metadata={'source': 'example.txt'}, similarity=0.041219234466552734, id=c8d801d0-9f50-412f-ac32-c6c6e4f7ff6a
什么是RAG..., metadata={'source': 'example.txt'}, similarity=0.041219234466552734, id=bddf5df9-0839-4e98-96b0-7d497790496d
什么是RAG..., metadata={'source': 'example.txt'}, similarity=0.04121926799416542, id=doc0
什么是RAG {'source': 'example.txt'} 0.041219234466552734
什么是RAG {'source': 'example.txt'} 0.041219234466552734
什么是RAG {'source': 'example.txt'} 0.04121926799416542


![1](mmr_search.png)

In [81]:
docs_mmr = db.max_marginal_relevance_search(
    query="RAG是什么",
    k=3,
    fetch_k=10,       # 内部先选 10 个候选
    lambda_mult=0.5   # 0.5 表示相关性与多样性平衡
)
print(docs_mmr)
for doc in docs_mmr:
    print(f"{doc.page_content[:50]}..., metadata={doc.metadata}")

[Document(id='c8d801d0-9f50-412f-ac32-c6c6e4f7ff6a', metadata={'source': 'example.txt'}, page_content='什么是RAG'), Document(id='bddf5df9-0839-4e98-96b0-7d497790496d', metadata={'source': 'example.txt'}, page_content='什么是RAG'), Document(id='doc1', metadata={'source': 'example.txt'}, page_content='RAG 是一种结合了检索和生成的自然语言处理技术。')]
什么是RAG..., metadata={'source': 'example.txt'}
什么是RAG..., metadata={'source': 'example.txt'}
RAG 是一种结合了检索和生成的自然语言处理技术。..., metadata={'source': 'example.txt'}


In [ ]:
""" 
本质上 retriever 是对底层检索函数的统一接口。核心还是 db.similarity_search
"""
query = "RAG是什么"
retriever = db.as_retriever(
    search_type="similarity",  # 检索策略 还有 mmr 等
    search_kwargs={"k":3,      # 作用是把参数传给底层检索函数 不传入就是默认 k fetch_k where（同filter） 
                   "filter": {"source": "example.txt"}}
)
docs = retriever.invoke(query)
for doc in docs:
    print(doc.page_content, doc.metadata, doc.id)

什么是RAG {'source': 'example.txt'} c8d801d0-9f50-412f-ac32-c6c6e4f7ff6a
什么是RAG {'source': 'example.txt'} bddf5df9-0839-4e98-96b0-7d497790496d
什么是RAG {'source': 'example.txt'} doc0
